In [ ]:
import pandas as pd
from pathlib import Path

df_cropyield = pd.read_pickle("data/raw/pickle/df_crop_yield_2016_2026.pkl")

# Optional: standardize column names
df_cropyield.columns = df_cropyield.columns.str.lower()

# View data
print(df_cropyield.shape)
df_cropyield.head()


In [ ]:
import pandas as pd

# Clean and create base year
df_cropyield["year_base"] = (
 df_cropyield["year"]
.astype(str)
.str.strip()
.str.split("-")
.str[0]
.astype(int)
)

# Filter rows for 2016 to 2022
df_cropyield_filtered = df_cropyield[
 df_cropyield["year_base"].between(2016, 2022)
].copy()

# Display result
print(df_cropyield_filtered.head())
print(df_cropyield_filtered.shape)

In [ ]:
# Group by the specified columns and calculate the average yield
df_cropyield_grouped = df_cropyield_filtered.groupby(
    ["district_name", "crop_name", "season", "year_base"], 
    as_index=False
)["yield"].mean()
df_cropyield_grouped


In [ ]:
# 1. Define the dictionary mapping synonyms to standard names
season_unification_map = {
    'Kharif': 'Kharif',
    'Autumn': 'Kharif',
    'Rabi': 'Rabi',
    'Winter': 'Rabi',
    'Summer': 'Summer',
    'Whole Year': 'Whole Year',
    'Total': 'Whole Year'
}

# 2. Map and update the season column directly
df_cropyield_grouped['season'] = df_cropyield_grouped['season'].map(season_unification_map)


In [ ]:
df_cropyield_grouped["season"].unique()

In [ ]:
# 1. Define the mapping of seasons to their respective month numbers
season_to_months = {
    'Kharif': [7, 8, 9, 10],
    'Rabi': [10, 11, 12, 1, 2, 3],
    'Summer': [3, 4, 5, 6],
    'Whole Year': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
}

# 2. Create the "month" column using map
df_cropyield_grouped['month'] = df_cropyield_grouped['season'].map(season_to_months)
df_cropyield_grouped

In [ ]:
df_cropyield_grouped = df_cropyield_grouped.rename(columns={"district_name": "district"})


In [ ]:
import pandas as pd
from pathlib import Path

df_precipitation = pd.read_pickle("data/processed/precipitation3.pkl")

# Optional: standardize column names
df_precipitation.columns = df_precipitation.columns.str.lower()

# View data
print(df_precipitation.head())
print(df_precipitation.shape)

In [ ]:
import pandas as pd

# 1. Clean up duplicate columns by keeping only the first instance
df_precipitation = df_precipitation.loc[:, ~df_precipitation.columns.duplicated()]

# 2. Now convert the single "date" column safely
df_precipitation["date"] = pd.to_datetime(df_precipitation["date"])

# 3. Extract year and month
df_precipitation["year"] = df_precipitation["date"].dt.year
df_precipitation["month"] = df_precipitation["date"].dt.month

# 4. Filter East Godavari data for years 2016 to 2022
df_precipitation_filtered = df_precipitation[
    df_precipitation["year"].between(2016, 2022)
].copy()

# View filtered dataframe
print(df_precipitation_filtered.head())
print(df_precipitation_filtered.shape)


In [ ]:
# Group by district, year (with trailing space), and month, then calculate the mean of precipitation_mm
df_precipitation_grouped = df_precipitation_filtered.groupby(
    ['district', 'year', 'month'], 
    as_index=False
)['precipitation_mm'].mean()

df_precipitation_grouped = df_precipitation_grouped.dropna(subset=["precipitation_mm"])


df_precipitation_grouped


In [ ]:
df_precipitation_grouped["precipitation_mm"].isna().sum()

In [ ]:
df_cropyield_grouped

In [ ]:
import pandas as pd

# 1. Create a safe row tracker column from the index (avoiding .reset_index() entirely)
df_cropyield_grouped['temp_row_id'] = df_cropyield_grouped.index

# 2. Explode the "month" column so each month in the list gets its own row
exploded = df_cropyield_grouped.explode('month')
exploded['month'] = exploded['month'].astype(int)

# 3. Calculate the actual calendar year (calendar_year) for Rabi and Summer months
def get_calendar_year(row):
    if row['season'] == 'Rabi' and row['month'] in [1, 2, 3]:
        return row['year_base'] + 1
    elif row['season'] == 'Summer' and row['month'] in [3, 4, 5, 6]:
        return row['year_base'] + 1
    return row['year_base']

exploded['calendar_year'] = exploded.apply(get_calendar_year, axis=1)

# 4. Merge the exploded table with df_precipitation_grouped
merged = pd.merge(
    exploded,
    df_precipitation_grouped,
    left_on=['district', 'calendar_year', 'month'],
    right_on=['district', 'year', 'month'],
    how='left'
)

# 5. Group back by our safe custom row ID and compute the average precipitation
avg_ppt_series = merged.groupby('temp_row_id')['precipitation_mm'].mean()

# 6. Assign the average precipitation back to your main DataFrame and clean up the tracker
df_cropyield_grouped['avg_ppt'] = df_cropyield_grouped['temp_row_id'].map(avg_ppt_series)
df_cropyield_grouped.drop(columns=['temp_row_id'], inplace=True)

df_cropyield_grouped


In [ ]:
df_cropyield_grouped["avg_ppt"].isna().sum()

In [ ]:
# Create df_cropyield_grouped_clean by removing NaN values from the avg_ppt column
df_cropyield_grouped_clean = df_cropyield_grouped.dropna(subset=['avg_ppt']).copy()
df_cropyield_grouped_clean

In [ ]:
df_cropyield_grouped_clean["crop_name"] = (
    df_cropyield_grouped_clean["crop_name"]
    .astype(str)
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

df_cropyield_grouped_clean

In [ ]:
df_cropyield_grouped_clean["crop_name"].unique()

In [ ]:
import pandas as pd

# Direct dictionary mapping: Crop Name -> Avg Net Profit (INR per Ton)
# (1 Ton = 10 Quintals)
crop_profit_per_ton = {
    'arhar/tur': 8500,
    'bajra': 3500,
    'banana': 12000,
    'barley': 4000,
    'coriander': 25000,
    'cotton(lint)': 8000,
    'dry_chillies': 60000,
    'garlic': 35000,
    'gram': 7500,
    'guar_seed': 9000,
    'jowar': 3800,
    'maize': 4500,
    'masoor': 8000,
    'moong(green_gram)': 9000,
    'onion': 4500,
    'peas_&_beans_(pulses)': 9500,
    'potato': 5000,
    'rapeseed_&mustard': 12000,
    'rice': 3500,
    'sannhamp': 4000,
    'sesamum': 15000,
    'sugarcane': 1350,  # Highly productive by tonnage, lower profit margin per individual ton
    'sweet_potato': 6000,
    'urad': 8500,
    'wheat': 5500,
    'cowpea(lobia)': 7500,
    'other_kharif_pulses': 6500,
    'other_rabi_pulses': 7000,
    'soyabean': 8000,
    'tapioca': 4500,
    'tobacco': 22000,
    'castor_seed': 9500,
    'ginger': 45000,
    'groundnut': 11000,
    'linseed': 8500,
    'moth': 7000,
    'other_oilseeds': 8000,
    'other_cereals': 3200,
    'other_summer_pulses': 7500,
    'safflower': 7500,
    'sunflower': 9000,
    'arecanut': 85000,
    'black_pepper': 120000,  # High-value export spice
    'cashewnut': 75000,
    'coconut': 15000,
    'ragi': 4000,
    'turmeric': 35000,
    'small_millets': 3500,
    'horse-gram': 6000,
    'niger_seed': 8500,
    'jute': 7000,
    'khesari': 5500,
    'mesta': 6000,
    'cardamom': 250000,      # Premium luxury crop
    'dry_ginger': 55000,
    'arcanut_(processed)': 110000,
    'cond-spcs_other': 25000,
    'mango': 25000,
    'other_fresh_fruits': 20000,
    'other_vegetables': 8000,
    'pineapple': 15000
}

# Clean backslashes from crop_name column if present in the data (e.g. 'peas\_&\_beans')
df_cropyield_grouped_clean['crop_name'] = df_cropyield_grouped_clean['crop_name'].str.replace('\\', '', regex=False)

# Map values directly to a new column
df_cropyield_grouped_clean['profit_per_ton_inr'] = df_cropyield_grouped_clean['crop_name'].map(crop_profit_per_ton)
df_cropyield_grouped_clean['profit']=df_cropyield_grouped_clean['profit_per_ton_inr']*df_cropyield_grouped_clean['yield']
df_cropyield_grouped_clean

In [ ]:

df_clean = df_cropyield_grouped_clean[
    (df_cropyield_grouped_clean["district"] == "Agra") & 
    (df_cropyield_grouped_clean["season"] == "Kharif")
]
df_clean

In [ ]:
import pandas as pd

# Define a function to filter rows in the top 20% of profit for each subgroup
def filter_top_percentile_crops(group, percentile_threshold=0.80):
    valid_profits = group['profit'].dropna()
    if not valid_profits.empty:
        # Calculate the 80th percentile threshold of profit for this specific group
        threshold = valid_profits.quantile(percentile_threshold)
        # Keep rows that are at or above the 80th percentile
        return group[group['profit'] >= threshold]
    else:
        return pd.DataFrame(columns=group.columns)

# Group by district and season, apply the filter, and reset the index
df_district_season_summary = (
    df_cropyield_grouped_clean
    .groupby(['district', 'season'], group_keys=False)
    .apply(filter_top_percentile_crops)
    .reset_index(drop=True)
)
df_district_season_summary

In [ ]:

df_clean1 = df_district_season_summary[
    (df_district_season_summary["district"] == "Agra") & 
    (df_district_season_summary["season"] == "Kharif")
]
df_clean1

In [ ]:
import pandas as pd
import numpy as np

# Find the index of the crop with the maximum profit in each district-season-year group
idx_best = (
    df_cropyield_grouped_clean
    .groupby(['district', 'season', 'year_base'])['profit']
    .idxmax()
)

# Extract only these best rows to serve as our ML training dataset
df_ml = df_cropyield_grouped_clean.loc[idx_best].copy()

# Rename crop_name to represent our target label: 'best_crop'
df_ml.rename(columns={'crop_name': 'best_crop'}, inplace=True)

# Print unique best crops and count of samples to verify progress
print(f"Dataset prepared! Total training rows: {len(df_ml)}")
print(f"Unique recommended crops: {df_ml['best_crop'].nunique()}")
print(df_ml['best_crop'].value_counts().head(5))


In [1]:
import os
import sys
from pyspark.sql import SparkSession

# Force Spark to use the exact Python executable running this VS Code notebook
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Force Spark to bypass corporate VPNs, domain name lookup, and port hijackings
spark = (
    SparkSession.builder
    .appName("CropRecommendation_Scale")
    .master("local[1]") # Uses 1 core; extremely stable in local debuggers
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")  # Disables port 4040 UI to avoid VS Code port conflicts
    .config("spark.network.timeout", "30s") # Stop hanging; throw an error if JVM is not responding
    .getOrCreate()
)

print("✅ Spark Session successfully bound and running in VS Code!")


KeyboardInterrupt: 

In [ ]:
import os
from pyspark.sql import SparkSession

# 1. Cleanly initialize Spark pinned to localhost (bypasses network/VPN DNS hangs)
spark = (
    SparkSession.builder
    .appName("CropRecommendation_Scale")
    .master("local[2]")  # Use 2 dedicated local worker threads
    .config("spark.driver.memory", "2g")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

# 2. Select clean columns from df_ml
cols_for_spark = ['district', 'season', 'avg_ppt', 'best_crop']
df_pandas_clean = df_ml[cols_for_spark].copy()

# 3. Export to a temporary local CSV
temp_csv_path = os.path.abspath("temp_crop_data_for_spark.csv")
df_pandas_clean.to_csv(temp_csv_path, index=False)

# 4. Read into Spark and CACHE into RAM immediately
spark_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(temp_csv_path)
    .cache()  # Keep in memory
)

# 5. Trigger an action so Spark loads all partitions BEFORE deleting the file
record_count = spark_df.count()
print(f"✅ Successfully loaded {record_count} records into Spark memory!")

# 6. NOW delete the temporary file safely
if os.path.exists(temp_csv_path):
    os.remove(temp_csv_path)

# 7. Print schema and preview
spark_df.printSchema()
spark_df.show(5, truncate=False)


In [ ]:
import pyspark
print("PySpark version:", pyspark.__version__)

from pyspark.sql import SparkSession
spark_test = SparkSession.builder.master("local[1]").appName("test").getOrCreate()
print("JVM started successfully!")


In [ ]:
import subprocess
try:
    print(subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT).decode())
except Exception as e:
    print("Java Error / Not Found:", e)


In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

# 1. Define StringIndexers for categorical features and the target label
# handleInvalid="keep" ensures Spark doesn't crash if it sees an unseen category during testing
district_indexer = StringIndexer(
    inputCol="district", 
    outputCol="district_indexed", 
    handleInvalid="keep"
)

season_indexer = StringIndexer(
    inputCol="season", 
    outputCol="season_indexed", 
    handleInvalid="keep"
)

crop_indexer = StringIndexer(
    inputCol="best_crop", 
    outputCol="label",  # Spark ML expects the target column to be named 'label' by default
    handleInvalid="keep"
)

# 2. Define the VectorAssembler to combine input features into a single 'features' vector
assembler = VectorAssembler(
    inputCols=["district_indexed", "season_indexed", "avg_ppt"],
    outputCol="features"
)

# 3. Fit the preprocessing steps on the Spark DataFrame and transform the data
preprocessing_pipeline = Pipeline(stages=[district_indexer, season_indexer, crop_indexer, assembler])
preprocessor_model = preprocessing_pipeline.fit(spark_df)
spark_df_prepared = preprocessor_model.transform(spark_df)

# Save the fitted target indexer so we can decode predicted numbers back to crop names later
fitted_crop_indexer = preprocessor_model.stages[2]

# 4. Preview the transformed Spark DataFrame
print("✅ Features indexed and assembled into vectors!")
spark_df_prepared.select(
    "district", "district_indexed", 
    "season", "season_indexed", 
    "avg_ppt", "features", 
    "best_crop", "label"
).show(5, truncate=False)


In [ ]:
!conda install -c conda-forge openjdk=11 -y

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

# 1. Perform a distributed random split (70% Train, 15% Validation, 15% Test)
# We set a seed so the split is perfectly reproducible across your Spark cluster
train_df, val_temp_df, test_temp_df = spark_df_prepared.randomSplit([0.70, 0.15, 0.15], seed=42)

# Cache the training set in memory to speed up the iterative random forest training process
train_df.cache()

# 2. Initialize the Spark-native Random Forest Classifier
# By default, Spark ML expects the input features to be in 'features' 
# and the target categories to be in 'label' (which we defined in Block 3)
rf_classifier = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,          # Parallelized training of 100 decision trees
    seed=42
)

# 3. Fit (Train) the model across the Spark cluster
print("Training the Distributed Spark Random Forest model...")
spark_model = rf_classifier.fit(train_df)

print("✅ Spark Random Forest Model training complete! 🎉")
print(f"Total training partition records: {train_df.count()}")


In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import IndexToString

# 1. Generate predictions on Validation and Testing sets
# .transform() generates columns like 'prediction', 'rawPrediction', and 'probability'
val_predictions = spark_model.transform(val_temp_df)
test_predictions = spark_model.transform(test_temp_df)


# 2. Initialize the Multiclass Evaluator
# Standard Top-1 accuracy evaluator
evaluator_top1 = MulticlassClassificationEvaluator(
    labelCol="label", 
    predictionCol="prediction", 
    metricName="accuracy"
)

# Top-3 and Top-5 accuracy evaluators (standard in Spark ML's multiclass evaluator)
evaluator_top3 = MulticlassClassificationEvaluator(
    labelCol="label", 
    predictionCol="probability", 
    metricName="topKAccuracy"
).setProbabilityCol("probability")

evaluator_top3.setParams(k=3)

evaluator_top5 = MulticlassClassificationEvaluator(
    labelCol="label", 
    predictionCol="probability", 
    metricName="topKAccuracy"
).setParams(k=5)


# 3. Compute Accuracies across your Spark cluster
val_accuracy = evaluator_top1.evaluate(val_predictions)
test_accuracy = evaluator_top1.evaluate(test_predictions)

test_accuracy_top3 = evaluator_top3.evaluate(test_predictions)
test_accuracy_top5 = evaluator_top5.evaluate(test_predictions)


# 4. Print Accuracy Performance Summary
print("=====================================================================")
print("             SPARK ML: DISTRIBUTED ACCURACY METRICS                 ")
print("=====================================================================")
print(f"Dataset      |   Standard Top-1 Accuracy   |   ⭐ Top-3 Recommendation")
print(f"-------------|-----------------------------|--------------------------------")
print(f"Validation   |            {val_accuracy * 100:.2f}%            |             -")
print(f"Testing      |            {test_accuracy * 100:.2f}%            |             {test_accuracy_top3 * 100:.2f}%")
print("---------------------------------------------------------------------")
print(f"⭐ Top-5 Testing Set Recommendation Accuracy: {test_accuracy_top5 * 100:.2f}%")
print("=====================================================================")


# 5. Decode predicted indices back to readable crop names (standard practice for end-user reports)
converter = IndexToString(
    inputCol="prediction", 
    outputCol="predicted_crop_name", 
    labels=fitted_crop_indexer.labels
)
test_predictions_with_names = converter.transform(test_predictions)

# Show a preview of the actual vs. predicted crops
print("\nPreview of Distributed Predictions (Actual vs. Predicted Crops):")
print("-" * 75)
test_predictions_with_names.select(
    "district", "season", "avg_ppt", "best_crop", "predicted_crop_name"
).show(10, truncate=False)
